In [2]:
import pandas as pd
from os import listdir
from os.path import join
import json

In [3]:
COMPRESSED_FOLDER_PATH = join("../.outFiles/analysis/compressed")

SENATE_DOCS_FOLDER_PATH = join("../.outFiles/senate/documents")
senate_file_list = listdir(SENATE_DOCS_FOLDER_PATH)
df_list = []
for file_name in senate_file_list:
    if not file_name.endswith(".json") or file_name.endswith("image-list.json"):
        continue
    with open(join(SENATE_DOCS_FOLDER_PATH, file_name), "r") as in_file:
        tmp = json.loads(in_file.read())
    
    name_components = file_name.split("_")
    year = name_components[0]
    ln = name_components[1]
    fn = name_components[2]
    doc_id = name_components[3]

    df = pd.DataFrame(tmp)
    df["p_full_name"] = fn + " " + ln
    df["p_chamber"] = "S"
    df["tx_year"] = year
    df["doc_id"] = doc_id.removesuffix(".json")
    df["data_source"] = "SELF_SENATE"
    
    df_list.append(df)

senate_df = pd.concat(df_list)
senate_df = senate_df.rename(columns={
    "transactionDate": "tx_date",
    "assetName": "asset_name",
    "assetType": "asset_type",
    "actionType": "tx_type",
    "comment": "comments"
})

senate_df = senate_df.drop(columns=["tickerUrlList"])

to_save_path = join(COMPRESSED_FOLDER_PATH, "senate.csv")
senate_df.to_csv(to_save_path, index=False, header=True)

senate_df


,tx_date,owner,ticker,asset_name,asset_type,tx_type,amount,comments,p_full_name,p_chamber,tx_year,doc_id,data_source
0,11/11/2014,Spouse,MDLZ,"Mondelez International, Inc. (NASDAQ)",,Sale (Full),"$50,001 - $100,000",--,ROY BLUNT,S,2014,9ddbcc76-dc18-4775-a7d5-a1a7063c0ebd,SELF_SENATE
0,04/08/2014,Self,AMT,American Tower Corporation (NYSE),,Sale (Full),"$15,001 - $50,000",--,CORY-A BOOKER,S,2014,29d797a6-e3ff-4d76-9ee9-2e3840adb15b,SELF_SENATE
1,04/08/2014,Self,NFLX,"Netflix, Inc. (NASDAQ)",,Sale (Full),"$15,001 - $50,000",--,CORY-A BOOKER,S,2014,29d797a6-e3ff-4d76-9ee9-2e3840adb15b,SELF_SENATE
0,08/08/2014,Self,NKE,"Nike, Inc. (NYSE)",,Sale (Full),"$1,001 - $15,000",--,CORY-A BOOKER,S,2014,7abb2400-6528-4f7f-9739-248dfedc3ca2,SELF_SENATE
1,08/08/2014,Self,IRM,Iron Mountain Inc. (NYSE),,Sale (Full),"$1,001 - $15,000",--,CORY-A BOOKER,S,2014,7abb2400-6528-4f7f-9739-248dfedc3ca2,SELF_SENATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,12/14/2023,Spouse,SNOW,Snowflake Inc Cl A,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,S,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE
1,12/14/2023,Spouse,LLY,Eli Lilly and Company,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,S,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE
2,12/07/2023,Self,TGT,Target Corp,Stock,Sale (Full),"$15,001 - $50,000",--,SHELDON WHITEHOUSE,S,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE
3,12/07/2023,Self,KO,Coca-Cola Company,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,S,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE


In [4]:
HOUSE_DOCS_FOLDER_PATH = join("../.outFiles/house/ocr_out/info_extract")
house_file_list = listdir(HOUSE_DOCS_FOLDER_PATH)
df_list = []
for file_name in house_file_list:
    if not file_name.endswith(".json"):
        continue
    with open(join(HOUSE_DOCS_FOLDER_PATH, file_name), "r") as in_file:
        tmp = json.loads(in_file.read())
    if len(tmp):
        name_components = file_name.split("_")
        year = name_components[0]
        district = name_components[1]
        ln = name_components[2]
        fn = name_components[3]
        doc_id = name_components[4]

        df = pd.DataFrame(tmp)
        df["p_full_name"] = fn + " " + ln
        df["p_chamber"] = "H"
        df["p_district"] = district
        df["tx_year"] = year
        df["doc_id"] = doc_id.removesuffix(".json")
        df["data_source"] = "SELF_HOUSE"

        df_list.append(df)

house_df = pd.concat(df_list)
house_df = house_df.reset_index(drop=True)

to_save_path = join(COMPRESSED_FOLDER_PATH, "house.csv")
house_df.to_csv(to_save_path, index=False, header=True)

house_df



,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,p_full_name,p_chamber,p_district,tx_year,doc_id,data_source
0,DECATUR ALA CITY BRD ED SPL TAX SCH WTS,None,None,P,JT,07/3/2014,07/3/2014,$15001 - $50000,FILING STATUS NEW,MR-MO BROOKS,H,AL05,2014,20000606,SELF_HOUSE
1,ETOWAH CNTY ALA BRD ED CAP OUTLAY WTS,None,None,P,JT,04/11/2014,04/11/2014,$1001 - $15000,FILING STATUS NEW,MR-MO BROOKS,H,AL05,2014,20000606,SELF_HOUSE
2,MOBILE COUNTY ALA BRD SCH COMMRSCAP OUTLAY WTS,None,None,P,JT,04/16/2014,04/16/2014,$1001 - $15000,FILING STATUS NEW,MR-MO BROOKS,H,AL05,2014,20000606,SELF_HOUSE
3,MORGAN STANLEY CAP TR V GTD CAP SECS (MWO),MWO,None,S,JT,05/5/2014,05/5/2014,$1001 - $15000,FILING STATUS NEW,MR-MO BROOKS,H,AL05,2014,20000606,SELF_HOUSE
4,PHENIX CITY AL SCH WTS GENL OBLIG- AT,None,None,P,JT,03/28/2014,03/28/2014,$15001 - $50000,FILING STATUS NEW,MR-MO BROOKS,H,AL05,2014,20000606,SELF_HOUSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,PRINCE GEORGES CNTY MD GO CONSOLIDATED 5.00 DU...,None,GS,S,JT,02/01/2024,02/01/2024,$500001 - $1000000,FILING STATUS NEW,SUZAN-K DELBENE,H,WA01,2024,20024495,SELF_HOUSE
0,BRISTOL-MYERS SQUIBB COMPANY (BMY) | ST | (BMY),BMY,ST,S,None,01/09/2024,01/12/2024,$1001 - $15000,FILING STATUS NEW SUBHOLDING OF RICHARD R LARS...,RICK LARSEN,H,WA02,2024,20024339,SELF_HOUSE
1,COLGATE-PALMOLIVE COMPANY (CL) | ST |,CL,ST,P,None,01/09/2024,01/12/2024,$1001 - $15000,FILING STATUS NEW SUBHOLDING OF RICHARD R LARS...,RICK LARSEN,H,WA02,2024,20024339,SELF_HOUSE
2,THE HERSHEY COMPANY (HSY) | ST |,HSY,ST,S,None,01/09/2024,01/12/2024,$1001 - $15000,FILING STATUS NEW SUBHOLDING OF RICHARD R LARS...,RICK LARSEN,H,WA02,2024,20024339,SELF_HOUSE
